# p234 Palindrome Linked List 解析笔记

- 题号：p234
- 题目英文名：Palindrome Linked List
- 题目中文名：回文链表
- 题目链接：https://leetcode.cn/problems/palindrome-linked-list/
- 题型：链表 / 快慢指针
- 难度：Easy
- 推荐优先级：中
- Java 原代码位置：`solutions-java/leetcode/p234_palindrome_linked_list/PalindromeLinkedList.java`


## 1. 题目一句话总结

这道题要求我们判断一条链表是否是回文结构，也就是正着看和反着看是否一致。

本质上考的是如何在链表上做对称比较，而 Java 原方案选择的是“找中点 + 原地反转后半段 + 比较 + 恢复链表”。


## 2. 题目理解与约束分析

### 2.1 题目要求

给定一条单链表，判断它是不是回文链表。

回文的意思是：

- 从左往右读
- 从右往左读

两边完全一致。

### 2.2 输入与输出

- 输入：链表头节点 `head`
- 输出：布尔值 `True/False`
- 返回结果含义：链表是否具有回文结构

### 2.3 关键约束

- 链表可能为空或只有一个节点
- 尽量使用 `O(1)` 额外空间
- 如果修改了链表结构，最好恢复回原样

### 2.4 示例分析

输入：`1 -> 2 -> 2 -> 1`

左右对称一致，所以返回 `True`。

输入：`1 -> 2`

两边不同，所以返回 `False`。


## 3. Java 原代码

```java
package leetcode.p234_palindrome_linked_list;

public class PalindromeLinkedList {

    public static class ListNode {
        public int val;
        public ListNode next;
    }

    public static boolean isPalindrome(ListNode head) {
        if (head == null || head.next == null) {
            return true;
        }
        ListNode slow = head;
        ListNode fast = head;
        while (fast.next != null && fast.next.next != null) {
            slow = slow.next;
            fast = fast.next.next;
        }

        ListNode pre = slow;
        ListNode cur = slow.next;
        ListNode next = null;
        slow.next = null;
        while (cur != null) {
            next = cur.next;
            cur.next = pre;
            pre = cur;
            cur = next;
        }

        boolean result = true;
        ListNode left = head;
        ListNode right = pre;
        while (left != null && right != null) {
            if (left.val != right.val) {
                result = false;
                break;
            }
            left = left.next;
            right = right.next;
        }

        cur = pre.next;
        pre.next = null;
        next = null;
        while (cur != null) {
            next = cur.next;
            cur.next = pre;
            pre = cur;
            cur = next;
        }
        return result;
    }
}
```


## 4. 先从 Java 原方案理解

Java 原解法非常完整，不只是判断回文，还把链表在比较后恢复了回来。

整体分 4 步：

1. 用快慢指针找到中点
2. 从中点之后开始，把后半段链表原地逆序
3. 左右两边同时向中间比较
4. 再把后半段恢复回原来的方向

其中一个很关键的细节是：

```java
slow.next = null;
```

Java 注释明确强调了这一句的作用：先把前半段和后半段断开，避免反转时形成循环链表。


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

最容易想到的办法是把链表复制到数组里，再用双指针判断数组是否回文。

### 5.2 为什么不够好

- 需要 `O(n)` 额外空间
- 没利用链表结构本身

### 5.3 优化方向

既然链表中点前后天然对应，那我们只需要把后半段逆过来，就能像数组双指针一样比较。Java 原方案正是这样做的，而且最后还恢复了结构。


## 6. 核心算法讲解

### 6.1 算法名称

- 快慢指针找中点
- 链表后半段原地反转

### 6.2 为什么想到这个算法

因为回文结构的本质就是左右对称。链表不能逆向随机访问，所以 Java 原解通过反转后半段，把“从右往左看”变成“从左往右走”。

### 6.3 关键状态

- `slow`、`fast`：定位中点
- `pre`、`cur`、`next_node`：反转后半段时使用
- `left`、`right`：比较左右两边

### 6.4 步骤拆解

1. 快慢指针找到中点 `slow`
2. 从 `slow.next` 开始逆序后半段
3. 左指针从头出发，右指针从逆序后的尾端出发，逐个比较
4. 比较结束后，再把链表逆回去


## 7. 过程演示

以 `1 -> 2 -> 2 -> 1` 为例：

```text
找中点后：
1 -> 2 | 2 -> 1

断开并逆序右半段后：
左边：1 -> 2
右边：1 -> 2

逐个比较：
1 == 1
2 == 2
```

所以结果是回文。接着再把右半段反转回去，恢复链表。


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional


@dataclass
class ListNode:
    val: int
    next: Optional["ListNode"] = None


def build_linked_list(values: list[int]) -> Optional[ListNode]:
    dummy = ListNode(0)
    cur = dummy
    for value in values:
        cur.next = ListNode(value)
        cur = cur.next
    return dummy.next


def linked_list_to_list(head: Optional[ListNode]) -> list[int]:
    ans: list[int] = []
    while head is not None:
        ans.append(head.val)
        head = head.next
    return ans


In [ ]:
class Solution:
    def isPalindrome(self, head: Optional[ListNode]) -> bool:
        if head is None or head.next is None:
            return True

        slow = head
        fast = head
        while fast.next is not None and fast.next.next is not None:
            slow = slow.next
            fast = fast.next.next

        pre = slow
        cur = slow.next
        next_node = None
        slow.next = None
        while cur is not None:
            next_node = cur.next
            cur.next = pre
            pre = cur
            cur = next_node

        result = True
        left = head
        right = pre
        while left is not None and right is not None:
            if left.val != right.val:
                result = False
                break
            left = left.next
            right = right.next

        cur = pre.next
        pre.next = None
        while cur is not None:
            next_node = cur.next
            cur.next = pre
            pre = cur
            cur = next_node

        return result


## 8. 代码逐段讲解

### 8.1 找中点

Java 原解里 `slow` 和 `fast` 都从头开始，循环结束后：

- 偶数长度时，`slow` 停在前半段最后一个节点
- 奇数长度时，`slow` 停在正中间节点

### 8.2 逆序后半段

逆序时不是从 `slow` 开始，而是从 `slow.next` 开始。同时先执行 `slow.next = null`，避免形成环。

### 8.3 左右比较

此时 `left` 从左半段头部出发，`right` 从逆序后的右半段头部出发，逐个比较即可。

### 8.4 恢复链表

Java 原方案最后又做了一次逆序，把链表恢复回原状。这一步非常加分，因为它让函数更稳健，不会把输入结构永久改坏。


## 9. 复杂度分析

- 时间复杂度：`O(n)`
- 为什么是这个时间复杂度：找中点、反转、比较、恢复都只是线性遍历
- 空间复杂度：`O(1)`
- 为什么是这个空间复杂度：只用了常数个指针变量


## 10. 易错点

1. 反转后半段前忘记断开 `slow.next`，可能形成环。
2. 比较完成后没恢复链表，导致调用方拿到被破坏的结构。
3. 中点位置处理不清，奇偶长度时容易错一位。
4. 把值相等判断写对了，但左右半段起点取错了。


## 11. 面试时怎么讲

可以这样概括：

> 这题我会先用快慢指针找到链表中点，再把后半段链表原地反转。
> 这样左半段和右半段就都能从左往右走，逐个比较即可。
> 比较完之后，我会再把后半段恢复回去，这和 Java 原方案保持一致。
> 所以整体时间复杂度 `O(n)`，额外空间复杂度 `O(1)`。


## 12. 实际应用场景

这题可以类比成：判断一条链式记录是否具有镜像对称结构。

### 具体业务案例：回文操作序列校验

#### 业务背景

某些系统里，一段链式操作记录可能需要满足前后对称，比如先加再减、先开再关之类的成对结构。

#### 输入是什么

输入是一条按时间串联的操作链。

#### 算法介入点

系统需要判断这条链式记录是否前后对称。

#### 输出是什么

输出是一个布尔值，表示这条结构是否回文。

#### 价值是什么

它解决的是链式结构的对称性校验问题，而且不依赖额外数组。


In [ ]:
head = build_linked_list([1, 2, 2, 1])
print('回文示例:', Solution().isPalindrome(head), linked_list_to_list(head))

head = build_linked_list([1, 2])
print('非回文示例:', Solution().isPalindrome(head), linked_list_to_list(head))


## 13. Demo 输出说明

- 第一组应输出 `True`，并且链表仍保持 `[1, 2, 2, 1]`，说明判断正确且结构被恢复。
- 第二组应输出 `False`，说明不对称结构能被识别出来。


## 14. 一句话复盘

> 这题最关键的不是找到中点，而是像 Java 原解那样在比较后把链表恢复回去。
